# MRCD source test notebook

Notebook này kiểm thử code trong `src` theo một luồng gần với pipeline thật:

- kiểm thử helper và tiền xử lý dữ liệu
- trích xuất thực thể và query
- bổ sung tri thức thực thể và tri thức fact-check từ internet
- lấy mẫu từ Bing news và từ kho tri thức
- tạo prompt cho LLM và kiểm tra parse nhãn
- dự đoán bằng LLM và SLM
- mô phỏng Round 1 và Round 2 của MRCD
- thử finetune RoBERTa với train, rồi dùng D_clean giả lập từ test để finetune lại

In [1]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Replace "YOUR_HF_TOKEN" with your actual token
login(token=user_secrets.get_secret("HF_TOKEN"))
print("login")

login


In [5]:
import os

GITHUB_REPO = "https://github.com/Chinh-de/Fake-news-detection.git"  # CẬP NHẬT TẠI ĐÂY
REPO_NAME = "Fake-news-detection"

if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO}
%cd {REPO_NAME}

Cloning into 'Fake-news-detection'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 112 (delta 12), reused 110 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 723.76 KiB | 24.96 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/kaggle/working/Fake-news-detection/Fake-news-detection


In [8]:
!pip install -r requirements.txt --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.8 MB/s eta 0:00:00:00:01


In [ ]:
from pathlib import Path
import os
import random
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

REPO_ROOT = Path('/kaggle/working/Fake-news-detection')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import MODEL_PATH, TRAIN_CSV, VAL_CSV, TEST_CSV, KNOWLEDGE_MODE
from src.utils import set_seed, preprocess_text, clean_query, truncate_text
from src.labels import generate_demo_label, to_clean_demo_label, parse_llm_label
from src.prompts import (
    build_dual_extraction_prompt,
    build_entity_extraction_prompt,
    build_classification_prompt,
)
from src.retrieval.demo_retrieval import search_news, retrieve_demonstrations, load_news_corpus
from src.retrieval.knowledge_agent import (
    build_knowledge_bundle,
    format_entity_definitions,
    format_verified_reports,
)
from src.retrieval.knowledge_retrieval import (
    analyze_claim_entities_and_query,
    retrieve_fact_evidence,
    format_fact_knowledge,
)
from src.pipeline.evidence import (
    prefetch_query_context,
    build_evidence_bundle,
    assess_with_llm,
    retrieve_from_clean_pool,
)
from src.pipeline.selection import split_clean_noisy, finalize_remaining_noisy_with_slm
from src.pipeline.finetune import maybe_finetune_slm_on_clean
from src.slm.model import IntegratedSLM
from src.slm.dataset import load_data_from_csv
from src.evaluation.metrics import evaluate_and_plot

set_seed(42)

RAW_DATA_PATH = REPO_ROOT / 'dataset' / 'twitter15_16.csv'
PREPROCESSED_DATA_PATH = REPO_ROOT / 'dataset' / 'twitter15_16_preprocessed.csv'

def safe_call(name, fn, *args, **kwargs):
    print(f'\n=== {name} ===')
    try:
        result = fn(*args, **kwargs)
        print('OK')
        return result
    except Exception as exc:
        print(f'SKIP/ERROR: {exc}')
        return None

def preview_text(text, limit=350):
    text = str(text)
    return text if len(text) <= limit else text[:limit] + '...'

def load_local_sample_data():
    raw_df = pd.read_csv(RAW_DATA_PATH)
    pre_df = pd.read_csv(PREPROCESSED_DATA_PATH)
    return raw_df, pre_df

def get_sample_text(pre_df, raw_df):
    sample_pre = pre_df.sample(1, random_state=42).iloc[0]
    sample_raw = raw_df.sample(1, random_state=42).iloc[0]
    text = str(sample_pre.get('text', sample_raw.get('content', ''))).strip()
    label = int(sample_pre.get('label_id', sample_pre.get('label', 1)))
    return sample_pre, sample_raw, text, label

def local_train_test_split(pre_df, text_col='text', label_col='label_id', train_size=0.7, val_size=0.15, test_size=0.15):
    if abs((train_size + val_size + test_size) - 1.0) > 1e-6:
        raise ValueError('train_size + val_size + test_size must equal 1.0')
    train_df, temp_df = train_test_split(
        pre_df,
        test_size=(1 - train_size),
        random_state=42,
        stratify=pre_df[label_col],
    )
    val_fraction = test_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=val_fraction,
        random_state=42,
        stratify=temp_df[label_col],
    )
    train_texts = pd.concat([train_df[text_col], val_df[text_col]], ignore_index=True).astype(str).tolist()
    train_labels = pd.concat([train_df[label_col], val_df[label_col]], ignore_index=True).astype(int).tolist()
    test_texts = test_df[text_col].astype(str).tolist()
    test_labels = test_df[label_col].astype(int).tolist()
    return train_texts, train_labels, test_texts, test_labels, train_df, val_df, test_df

def _parse_timestamp_series(series):
    numeric_ts = pd.to_numeric(series, errors='coerce')
    if numeric_ts.notna().mean() >= 0.8:
        return pd.to_datetime(numeric_ts, unit='s', errors='coerce')
    return pd.to_datetime(series, errors='coerce')

def time_based_train60_split(pre_df, text_col='text', label_col='label_id', timestamp_col='timestamp', train_ratio=0.6):
    if timestamp_col not in pre_df.columns:
        raise ValueError(f'Missing timestamp column: {timestamp_col}')
    if text_col not in pre_df.columns or label_col not in pre_df.columns:
        raise ValueError('Missing required text/label columns')

    df = pre_df.copy()
    df['timestamp_dt'] = _parse_timestamp_series(df[timestamp_col])
    df = df.dropna(subset=['timestamp_dt', text_col, label_col]).sort_values('timestamp_dt').reset_index(drop=True)
    if len(df) < 10:
        raise ValueError('Not enough rows for time-based split')

    split_idx = max(1, int(len(df) * float(train_ratio)))
    train_df = df.iloc[:split_idx].copy()
    holdout_df = df.iloc[split_idx:].copy()

    train_texts = train_df[text_col].astype(str).tolist()
    train_labels = train_df[label_col].astype(int).tolist()
    holdout_texts = holdout_df[text_col].astype(str).tolist()
    holdout_labels = holdout_df[label_col].astype(int).tolist()

    return {
        'train_texts': train_texts,
        'train_labels': train_labels,
        'holdout_texts': holdout_texts,
        'holdout_labels': holdout_labels,
        'train_df': train_df,
        'holdout_df': holdout_df,
    }

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'KNOWLEDGE_MODE = {KNOWLEDGE_MODE}')
print(f'MODEL_PATH = {MODEL_PATH}')

TypeError: unsupported operand type(s) for /: 'str' and 'str'

## 1. Nạp dữ liệu mẫu

Cell này lấy một mẫu tin từ dữ liệu gốc và một mẫu từ bản tiền xử lý để kiểm tra đầu vào cho toàn bộ notebook.

In [ ]:
raw_df, pre_df = load_local_sample_data()
sample_pre, sample_raw, sample_text, sample_label = get_sample_text(raw_df=raw_df, pre_df=pre_df)

print('Raw columns:', list(raw_df.columns))
print('Preprocessed columns:', list(pre_df.columns))
print('Sample label_id:', sample_label)
print('Sample text preview:')
print(preview_text(sample_text, 500))

display(raw_df.head(2))
display(pre_df.head(2))

## 2. Test các helper cơ bản theo từng hàm

Mỗi hàm có 2 cell: 1 markdown mô tả và 1 code in ra `kết quả thực tế` và `kết quả mong đợi` để tự so sánh.

In [ ]:
print('Hàm: preprocess_text')
input_text = 'BREAKING: @CNN reports http://t.co/abc123 shocking news!!! #fakenews'
actual = preprocess_text(input_text)
expected = 'breaking reports shocking news #fakenews'

print('Input:')
print(input_text)
print('\nKết quả thực tế:')
print(actual)
print('\nKết quả mong đợi:')
print(expected)

preprocess_text: breaking reports shocking news #fakenews
clean_query: What exactly happened
truncate_text: hello world...
generate_demo_label: Objective
to_clean_demo_label(0): Real
to_clean_demo_label(1): Fake
parse_llm_label('Real answer'): (0, 'Real')
parse_llm_label('Fake answer'): (1, 'Fake')


### Test hàm clean_query

Kiểm tra loại bỏ dấu câu và chuẩn hóa khoảng trắng cho truy vấn.

In [ ]:
print('Hàm: clean_query')
input_query = 'What,   exactly, happened?!'
actual = clean_query(input_query)
expected = 'What exactly happened'

print('Input:')
print(input_query)
print('\nKết quả thực tế:')
print(actual)
print('\nKết quả mong đợi:')
print(expected)

### Test hàm truncate_text

Kiểm tra cắt chuỗi dài và thêm dấu `...` đúng kỳ vọng.

In [ ]:
print('Hàm: truncate_text')
input_text = 'hello world foo bar'
actual = truncate_text(input_text, max_length=12)
expected = 'hello world...'

print('Input:')
print(input_text)
print('\nKết quả thực tế:')
print(actual)
print('\nKết quả mong đợi:')
print(expected)

### Test hàm parse_llm_label

Kiểm tra ánh xạ đầu ra LLM thành nhãn nhị phân (Real -> 0, Fake -> 1).

In [ ]:
print('Hàm: parse_llm_label')
input_real = 'Real'
input_fake = 'Fake'
actual_real = parse_llm_label(input_real)
actual_fake = parse_llm_label(input_fake)
expected_real = 0
expected_fake = 1

print('Input 1:', input_real)
print('Kết quả thực tế:', actual_real)
print('Kết quả mong đợi:', expected_real)
print('\nInput 2:', input_fake)
print('Kết quả thực tế:', actual_fake)
print('Kết quả mong đợi:', expected_fake)

### Test hàm to_clean_demo_label

Kiểm tra chuyển nhãn nhị phân sang chuỗi nhãn trực tiếp dùng cho demo vòng sau.

In [ ]:
print('Hàm: to_clean_demo_label')
actual_0 = to_clean_demo_label(0)
actual_1 = to_clean_demo_label(1)
expected_0 = 'Real'
expected_1 = 'Fake'

print('Input: 0')
print('Kết quả thực tế:', actual_0)
print('Kết quả mong đợi:', expected_0)
print('\nInput: 1')
print('Kết quả thực tế:', actual_1)
print('Kết quả mong đợi:', expected_1)

### Test hàm split_clean_noisy

Kiểm tra logic tách mẫu clean/noisy theo điều kiện đồng thuận LLM-SLM và ngưỡng confidence.

In [ ]:
print('Hàm: split_clean_noisy')
sample_clean = {'label_llm': 1, 'label_slm': 1, 'conf_slm': 0.95}
sample_noisy = {'label_llm': 1, 'label_slm': 0, 'conf_slm': 0.95}
threshold = 0.8

actual_clean = split_clean_noisy(sample_clean, threshold)
actual_noisy = split_clean_noisy(sample_noisy, threshold)
expected_clean = True
expected_noisy = False

print('Case clean -> kết quả thực tế:', actual_clean, '| mong đợi:', expected_clean)
print('Case noisy -> kết quả thực tế:', actual_noisy, '| mong đợi:', expected_noisy)

### Test hàm retrieve_from_clean_pool

Kiểm tra truy xuất demo từ D_clean bằng BM25 và kiểm tra số lượng kết quả trả về.

In [ ]:
print('Hàm: retrieve_from_clean_pool')
clean_pool_demo = [
    {'text': 'nasa mission update confirms launch next month', 'label': 0},
    {'text': 'fabricated claim about celebrity event', 'label': 1},
]
query_demo = 'nasa mission update'
actual = retrieve_from_clean_pool(query_demo, clean_pool_demo, k=2)
expected = 'Danh sách có từ 1 đến 2 demo, mỗi demo có label Real/Fake từ D_clean'

print('Query:', query_demo)
print('Kết quả thực tế (số lượng):', len(actual))
print('Kết quả mong đợi:', expected)
print('\nChi tiết kết quả thực tế:')
for i, item in enumerate(actual, start=1):
    print(f'[{i}] label={item.get("label")}, source={item.get("source")}, text={preview_text(item.get("text", ""), 120)}')

### Test hàm load_data_from_csv (src/slm/dataset.py)

Kiểm tra hàm nạp dữ liệu chuẩn từ `src.slm.dataset` (đầu ra gồm train_texts, train_labels, test_texts, test_labels).

In [ ]:
print('Hàm src.slm.dataset.load_data_from_csv')
actual = None
expected = 'Tuple gồm 4 phần tử: train_texts, train_labels, test_texts, test_labels'

if all(Path(p).exists() for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]):
    train_texts_x, train_labels_x, test_texts_x, test_labels_x = load_data_from_csv(TRAIN_CSV, VAL_CSV, TEST_CSV)
    actual = {
        'train_texts': len(train_texts_x),
        'train_labels': len(train_labels_x),
        'test_texts': len(test_texts_x),
        'test_labels': len(test_labels_x),
    }
else:
    actual = {'skip': True, 'reason': 'Configured TRAIN/VAL/TEST CSV not found'}

print('Kết quả thực tế:')
print(actual)
print('\nKết quả mong đợi:')
print(expected)

### Test hàm maybe_finetune_slm_on_clean (src/pipeline/finetune.py)

Kiểm tra nhánh logic của hàm finetune: disabled và insufficient_samples, không cần chạy train thật.

In [ ]:
print('Hàm: maybe_finetune_slm_on_clean (không dùng dummy)')

time_split_real = time_based_train60_split(
    pre_df, text_col='text', label_col='label_id', timestamp_col='timestamp', train_ratio=0.6
)
train_texts_r = time_split_real['train_texts']
train_labels_r = time_split_real['train_labels']
holdout_texts_r = time_split_real['holdout_texts']
holdout_labels_r = time_split_real['holdout_labels']

MAX_TRAIN = min(256, len(train_texts_r))
MAX_HOLDOUT = min(128, len(holdout_texts_r))
train_texts_small_r = train_texts_r[:MAX_TRAIN]
train_labels_small_r = train_labels_r[:MAX_TRAIN]
holdout_texts_small_r = holdout_texts_r[:MAX_HOLDOUT]
holdout_labels_small_r = holdout_labels_r[:MAX_HOLDOUT]

slm_model_path = MODEL_PATH if Path(MODEL_PATH).exists() else 'roberta-base'
slm_real = IntegratedSLM(model_path=slm_model_path)

print('Đánh giá baseline trước khi finetune trên holdout...')
baseline_preds = slm_real.inference_batch(holdout_texts_small_r, batch_size=16)
baseline_y_pred = [pred for pred, conf, probs in baseline_preds]
baseline_metrics = evaluate_and_plot(
    y_true=holdout_labels_small_r,
    y_pred=baseline_y_pred,
    labels=['Real', 'Fake'],
    model_name='SLM baseline before train60 finetune',
)

print('Finetune SLM với 60% dữ liệu sớm nhất...')
finetune_stats_real = slm_real.fnetune(
    train_texts=train_texts_small_r,
    train_labels=train_labels_small_r,
    model_init=slm_model_path,
    epochs=1,
    batch_size=16,
    lr=1e-5,
    weight_decay=1e-4,
    save_path=None,
)
print('Finetune stats:', finetune_stats_real)

after_preds = slm_real.inference_batch(holdout_texts_small_r, batch_size=16)
after_y_pred = [pred for pred, conf, probs in after_preds]
after_metrics = evaluate_and_plot(
    y_true=holdout_labels_small_r,
    y_pred=after_y_pred,
    labels=['Real', 'Fake'],
    model_name='SLM after train60 finetune',
)

clean_pool_small = [
    {'text': t, 'label': y}
    for t, y, p in zip(holdout_texts_small_r, holdout_labels_small_r, after_y_pred)
    if int(y) == int(p)
][:32]

actual_disabled = maybe_finetune_slm_on_clean(
    slm=slm_real,
    clean_pool=clean_pool_small,
    round_id=2,
    enable_slm_finetune=False,
    slm_finetune_min_samples=4,
)

actual_insufficient = maybe_finetune_slm_on_clean(
    slm=slm_real,
    clean_pool=clean_pool_small[:1],
    round_id=2,
    enable_slm_finetune=True,
    slm_finetune_min_samples=4,
)

print('Kết quả thực tế - disabled case:')
print(actual_disabled)
print('\nKết quả thực tế - insufficient case:')
print(actual_insufficient)
print('\nKết quả mong đợi:')
print("disabled -> {'trained': False, 'reason': 'disabled'}")
print("insufficient -> {'trained': False, 'reason': 'insufficient_samples', ...}")


## 3. Test nhóm hàm trích xuất thực thể và tri thức

Phần này được tách theo từng hàm thành phần để dễ quan sát input, output thực tế và kỳ vọng.

In [ ]:
print('Hàm: build_entity_extraction_prompt')
actual = build_entity_extraction_prompt(sample_text)
expected = 'Prompt chứa schema JSON với key entities và không có giải thích thừa'

print('Kết quả thực tế (preview):')
print(preview_text(actual, 700))
print('\nKết quả mong đợi:')
print(expected)

NameError: name 'sample_text' is not defined

### Test hàm build_dual_extraction_prompt

Kiểm tra prompt kép có cả `entities` và `query`.

In [ ]:
print('Hàm: build_dual_extraction_prompt')
actual = build_dual_extraction_prompt(sample_text)
expected = 'Prompt chứa schema JSON với entities và query, kèm ràng buộc neutral query'

print('Kết quả thực tế (preview):')
print(preview_text(actual, 800))
print('\nKết quả mong đợi:')
print(expected)

### Nhóm test knowledge extraction (src/retrieval)

Các test bên dưới chạy theo thứ tự: analyze_claim_entities_and_query -> build_knowledge_bundle(wiki_only/full).

### Test hàm analyze_claim_entities_and_query

Kiểm tra trích xuất thực thể và query từ claim.

In [ ]:
print('Hàm: analyze_claim_entities_and_query')
RUN_HEAVY_SMOKE = os.getenv('RUN_HEAVY_SMOKE', '0') == '1'
expected = "Kết quả có key 'entities' (list) và 'query' (str)"

if RUN_HEAVY_SMOKE:
    actual = safe_call('analyze_claim_entities_and_query', analyze_claim_entities_and_query, sample_text, mode='full')
else:
    actual = {'skip': True, 'reason': 'Set RUN_HEAVY_SMOKE=1 to run heavy extraction'}

print('Kết quả thực tế:')
print(actual)
print('\nKết quả mong đợi:')
print(expected)

### Test hàm build_knowledge_bundle (wiki_only)

Kiểm tra tạo gói tri thức wiki_only từ thực thể.

In [ ]:
print('Hàm: build_knowledge_bundle(mode=wiki_only)')
RUN_HEAVY_SMOKE = os.getenv('RUN_HEAVY_SMOKE', '0') == '1'
expected = "Dict có key 'combined_text' và 'mode'='wiki_only'"

if RUN_HEAVY_SMOKE:
    actual = safe_call('build_knowledge_bundle wiki_only', build_knowledge_bundle, sample_text, mode='wiki_only')
else:
    actual = {'skip': True, 'reason': 'Set RUN_HEAVY_SMOKE=1 to run knowledge build'}

print('Kết quả thực tế:')
print({'keys': list(actual.keys()) if isinstance(actual, dict) else actual, 'mode': actual.get('mode') if isinstance(actual, dict) else None})
print('\nKết quả mong đợi:')
print(expected)

### Test hàm build_knowledge_bundle (full)

Kiểm tra tạo gói tri thức đầy đủ (`wiki + fact-check`), có thể nặng nên chạy có điều kiện.

In [ ]:
print('Hàm: build_knowledge_bundle(mode=full)')
RUN_HEAVY_SMOKE = os.getenv('RUN_HEAVY_SMOKE', '0') == '1'
expected = "Dict có key 'combined_text' và 'mode'='full'"

if RUN_HEAVY_SMOKE:
    actual = safe_call('build_knowledge_bundle full', build_knowledge_bundle, sample_text, mode='full')
else:
    actual = {'skip': True, 'reason': 'Set RUN_HEAVY_SMOKE=1 to run full knowledge build'}

print('Kết quả thực tế:')
print({'keys': list(actual.keys()) if isinstance(actual, dict) else actual, 'mode': actual.get('mode') if isinstance(actual, dict) else None})
print('\nKết quả mong đợi:')
print(expected)

## 4. Test nhóm hàm retrieval (từ nhỏ đến lớn)

Các test được tách riêng theo hàm: load corpus -> search -> retrieve demos -> retrieve fact evidence.

In [ ]:
print('Hàm: load_news_corpus')
static_corpus = safe_call('load_news_corpus', load_news_corpus)
if not static_corpus:
    static_corpus = pre_df['text'].astype(str).head(200).tolist()
actual = len(static_corpus)
expected = 'Số lượng > 0 (load được corpus hoặc fallback local)'

print('Kết quả thực tế:', actual)
print('Kết quả mong đợi:', expected)

NameError: name 'safe_call' is not defined

### Test hàm search_news

Kiểm tra truy xuất mẫu tin từ internet qua `src.retrieval.demo_retrieval.search_news`.

In [ ]:
print('Hàm: search_news')
RUN_HEAVY_DEMOS = os.getenv('RUN_HEAVY_DEMOS', '0') == '1'
expected = 'Danh sách text news, có thể rỗng khi không mạng'

if RUN_HEAVY_DEMOS:
    actual = safe_call('search_news', search_news, sample_text, max_results=3) or []
else:
    actual = ['SKIP: set RUN_HEAVY_DEMOS=1 to run internet search']

print('Kết quả thực tế (số lượng):', len(actual))
print('Mẫu kết quả thực tế:', preview_text(actual[0], 150) if actual else '[]')
print('Kết quả mong đợi:', expected)

### Test hàm retrieve_demonstrations

Kiểm tra truy xuất top-k demonstrations từ corpus bằng BM25.

In [ ]:
print('Hàm: retrieve_demonstrations')
if 'static_corpus' not in globals() or not static_corpus:
    static_corpus = pre_df['text'].astype(str).head(200).tolist()

query_demo = sample_text
demo_pool = static_corpus[:80]
actual = retrieve_demonstrations(query_demo, demo_pool, k=4)
expected = 'Danh sách tối đa 4 demo, mỗi demo có text/label/source'

print('Kết quả thực tế (số lượng):', len(actual))
print('Kết quả mong đợi:', expected)
print('\nChi tiết thực tế:')
for i, item in enumerate(actual, start=1):
    print(f'[{i}] keys={list(item.keys())}, label={item.get("label")}, source={item.get("source")}')

### Test hàm prefetch_query_context

Kiểm tra dựng ngữ cảnh truy xuất ban đầu cho claim.

In [ ]:
print('Hàm: prefetch_query_context')
RUN_HEAVY_DEMOS = os.getenv('RUN_HEAVY_DEMOS', '0') == '1'
expected = "Dict có key knowledge_text, bing_seed_news, knowledge_mode"

if RUN_HEAVY_DEMOS:
    actual = safe_call(
        'prefetch_query_context',
        prefetch_query_context,
        sample_text,
        demo_k=3,
        fact_top_k=2,
        reuse_knowledge_cache=False,
        knowledge_mode='wiki_only',
        wiki_fetch_full=False,
    )
else:
    actual = {'skip': True, 'reason': 'Set RUN_HEAVY_DEMOS=1 to run retrieval context'}

print('Kết quả thực tế:')
if isinstance(actual, dict):
    print({'keys': list(actual.keys()), 'knowledge_mode': actual.get('knowledge_mode')})
else:
    print(actual)
print('\nKết quả mong đợi:')
print(expected)

### Test hàm build_evidence_bundle

Kiểm tra tạo evidence bundle theo round từ các thành phần retrieval.

### Test hàm retrieve_fact_evidence

Kiểm tra pipeline retrieval đầy đủ (query analysis + trusted search + crawl + rerank), chạy có điều kiện vì tốn tài nguyên.

In [ ]:
print('Hàm: retrieve_fact_evidence')
RUN_HEAVY_DEMOS = os.getenv('RUN_HEAVY_DEMOS', '0') == '1'
expected = "Dict có key 'analysis' và 'top_chunks' (list)"

if RUN_HEAVY_DEMOS:
    actual = safe_call(
        'retrieve_fact_evidence',
        retrieve_fact_evidence,
        sample_text,
        max_urls=5,
        top_k_chunks=3,
        similarity_threshold=3.5,
    )
else:
    actual = {'skip': True, 'reason': 'Set RUN_HEAVY_DEMOS=1 to run internet+crawl+rereank'}

print('Kết quả thực tế:')
if isinstance(actual, dict):
    print({'keys': list(actual.keys()), 'top_chunks_len': len(actual.get('top_chunks', [])) if isinstance(actual.get('top_chunks', []), list) else None})
else:
    print(actual)
print('\nKết quả mong đợi:')
print(expected)

### Test hàm IntegratedSLM (init + inference)

Kiểm tra khởi tạo SLM và chạy inference 1 mẫu bằng `src.slm.model.IntegratedSLM`.

In [ ]:
print('Hàm: IntegratedSLM.__init__ + inference')
slm_model_path = MODEL_PATH if Path(MODEL_PATH).exists() else 'roberta-base'
expected = 'Khởi tạo được model và inference trả về (pred:int, conf:float, probs:array)'

actual = None
try:
    slm = IntegratedSLM(model_path=slm_model_path)
    pred, conf, probs = slm.inference(sample_text)
    actual = {
        'pred': int(pred),
        'conf': float(conf),
        'probs_len': len(probs),
    }
except Exception as exc:
    actual = {'error': str(exc)}

print('Kết quả thực tế:')
print(actual)
print('\nKết quả mong đợi:')
print(expected)